In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "../data/processed/house_price_processed.csv"
)

print(df.shape)
df.head()

(14620, 34)


,id,Date,number of bedrooms,number of bathrooms,living area,lot area,number of floors,waterfront present,number of views,condition of the house,...,Sale_Year,Sale_Month,Sale_Quarter,Property_Age,Renovated,Basement_Percentage,schools_norm,airport_norm,grade_norm,Infrastructure_Score
0,6762810145,42491,5,2.50,3650,9050,2.0,0,4,5,...,2016,5,2,104,0,7.671233,0.5,0.733333,0.666667,0.620000
1,6762810635,42491,4,2.50,2920,4000,1.5,0,0,5,...,2016,5,2,116,0,34.589041,0.5,0.966667,0.444444,0.623333
2,6762810998,42491,5,2.75,2910,9480,1.5,0,0,3,...,2016,5,2,86,0,0.000000,0.0,0.900000,0.444444,0.403333
3,6762812605,42491,4,2.50,3310,42998,2.0,0,0,3,...,2016,5,2,24,0,0.000000,1.0,0.133333,0.555556,0.606667
4,6762812919,42491,3,2.00,2710,4500,1.5,0,0,4,...,2016,5,2,96,0,30.627306,0.0,0.966667,0.444444,0.423333


Normalize components

In [2]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

df["Age_Risk"] = scaler.fit_transform(
    df[["Property_Age"]]
)

df["Condition_Risk"] = 1 - scaler.fit_transform(
    df[["condition of the house"]]
)

df["Grade_Risk"] = 1 - scaler.fit_transform(
    df[["grade of the house"]]
)

df["Infrastructure_Risk"] = 1 - scaler.fit_transform(
    df[["Infrastructure_Score"]]
)

Risk score

In [3]:
df["Risk_Score"] = (
    0.35 * df["Age_Risk"]
    + 0.25 * df["Condition_Risk"]
    + 0.25 * df["Grade_Risk"]
    + 0.15 * df["Infrastructure_Risk"]
) * 100

Risk Categories

In [4]:
df["Risk_Category"] = pd.cut(
    df["Risk_Score"],
    bins=[0, 33, 66, 100],
    labels=[
        "Low Risk",
        "Medium Risk",
        "High Risk"
    ]
)

Verification

In [5]:
df["Risk_Category"].value_counts()

Risk_Category
Medium Risk    12426
Low Risk        1542
High Risk        652
Name: count, dtype: int64

In [6]:
df[
    [
        "Risk_Score",
        "Risk_Category"
    ]
].head()

,Risk_Score,Risk_Category
0,42.548090,Medium Risk
1,51.705314,Medium Risk
2,58.408213,Medium Risk
3,33.680061,Medium Risk
4,54.898661,Medium Risk


In [7]:
df.to_csv(
    "../data/processed/house_price_with_risk.csv",
    index=False
)

# Recommendation Engine

Create Investment Score

In [10]:


scaler = MinMaxScaler()

df["Grade_Score"] = scaler.fit_transform(
    df[["grade of the house"]]
)

df["View_Score"] = scaler.fit_transform(
    df[["number of views"]]
)

df["Investment_Score"] = (
    0.40 * df["Grade_Score"]
    + 0.30 * df["View_Score"]
    + 0.30 * (1 - df["Risk_Score"]/100)
) * 100

generate recommendation

In [11]:
def investment_recommendation(row):

    if (
        row["Risk_Score"] <= 30
        and row["Investment_Score"] >= 70
    ):
        return "BUY"

    elif (
        row["Risk_Score"] <= 60
        and row["Investment_Score"] >= 40
    ):
        return "HOLD"

    else:
        return "SELL"


df["Recommendation"] = df.apply(
    investment_recommendation,
    axis=1
)

Checking distribution

In [12]:
df["Recommendation"].value_counts()

Recommendation
SELL    11365
HOLD     3163
BUY        92
Name: count, dtype: int64

In [13]:
df.to_csv(
    "../data/processed/house_price_with_risk_recommendation.csv",
    index=False
)